In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/05_model_input/AR6_Scenarios_with_fuel_intensity.csv")
df = df.loc[df["sector"] == "Power",:]
df = df.loc[df["technology"].isin([ 'CoalCap', 'GasCap', 'CoalCap - w/o CCS',
       'GasCap - w/o CCS' ]),:]

/var/folders/df/zghzv05d7xb8t9xy7y5ld_h40000gn/T/ipykernel_54137/2594637895.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/05_model_input/AR6_Scenarios_with_fuel_intensity.csv")


# Compatible scenarios

In [ ]:
# the EBITDA is likely to be negative when the values in fom_usd_per_mw_yr are higher
#  than capacity_factor * hours_per_year * scenario_price
incompatible_fixed_cost = df["om_cost_usd_per_mw_per_yr"] > df["scenario_capacity_factor"] * (24*365) * df["scenario_price"]

# the EBITDA is likely to be negative when fuel_price/efficiency > scenario_price .
# incompatible_var_cost = df["fuel_price"] / df["efficiency_decimal"] > (df["scenario_price"]*2)
incompatible_var_cost = df["fuel_price"] * df["fuel_intensity"] > df["scenario_price"]


In [4]:
df = df.assign(
    incompatible_var_cost = incompatible_var_cost.astype(bool),
    incompatible_fixed_cost = incompatible_fixed_cost.astype(bool),
)



In [5]:
df.loc[df["scenario_type"] == "baseline", ["scenario_provider", "scenario"]].drop_duplicates()

,scenario_provider,scenario
22031,AIM/CGE 2.2,EN_NPi2020_1200f
68017,COFFEE 1.1,CO_CurPol
111083,COFFEE 1.1,EN_NPi2020_1000_COV
134184,COFFEE 1.1,EN_NPi2020_600_COV
262970,IMACLIM 1.1,ADVANCE_NoPolicy_WP6
271954,IMACLIM 1.1,SSP3_NoPolicy_TranspBase
388896,IMAGE 3.2,SSP1-baseline
420338,IMAGE 3.2,SSP2-baseline
468901,MESSAGEix-GLOBIOM_1.0,CO_CurPol
629218,MESSAGEix-GLOBIOM_1.2,COV_GreenPush


In [6]:
pd.options.display.max_rows = 1000

incompatibility_flagged = df[[
    "scenario_provider", "scenario", "scenario_type","sector", "technology",  "scenario_geography",
    "incompatible_var_cost","incompatible_fixed_cost"]].drop_duplicates()


incompatibility_flagged = incompatibility_flagged.groupby(["scenario_provider", "scenario"]).agg(
    n_techs = ("technology", "nunique"),
    n_regions = ("scenario_geography", "nunique"),
    incompatible_var_cost = ("incompatible_var_cost", "any"),
    incompatible_fixed_cost = ("incompatible_fixed_cost", "any"),
)

incompatibility_flagged = incompatibility_flagged.assign(
    likely_incompatible_scenario = (incompatibility_flagged["incompatible_var_cost"] | incompatibility_flagged["incompatible_fixed_cost"]),
    incompatible_scenario = (incompatibility_flagged["incompatible_var_cost"] & incompatibility_flagged["incompatible_fixed_cost"]),
)


incompatibility_flagged.query("(incompatible_var_cost == False) & (incompatible_fixed_cost == False)")

n_techs  \
scenario_provider             scenario                                                    
AIM/CGE 2.2                   EN_INDCi2030_1200f                                      4   
                              EN_INDCi2030_1400                                       4   
                              EN_INDCi2030_1400f                                      4   
                              EN_INDCi2030_1600                                       4   
                              EN_INDCi2030_1600f                                      4   
                              EN_INDCi2030_1800                                       4   
                              EN_INDCi2030_1800f                                      4   
                              EN_INDCi2030_800f                                       4   
                              EN_INDCi2030_900f                                       4   
                              EN_NPi2020_1000f                                        4   
                              EN_NPi2020_1200                                         4   
                              EN_NPi2020_1200f                                        4   
                              EN_NPi2020_1400                                         4   
                              EN_NPi2020_1400f                                        4   
                              EN_NPi2020_1600                                         4   
                              EN_NPi2020_1600f                                        4   
                              EN_NPi2020_1800                                         4   
                              EN_NPi2020_1800f                                        4   
                              EN_NPi2020_600                                          4   
                              EN_NPi2020_900f                                         4   
                              EN_NPi2100                                              4   
AIM/CGE-Korea 2.1             CO_2Deg2020                                             4   
                              CO_2Deg2030                                             4   
                              CO_BAU                                                  4   
                              CO_Bridge                                               4   
                              CO_CurPol                                               4   
                              CO_GPP                                                  4   
                              CO_LowCarbon                                            4   
                              CO_NDCMCS                                               4   
AIM/Enduse-Japan 2.1          CDLINKS-NDC                                             4   
                              CDLINKS-NPi                                             4   
                              CDLINKS-NPi1600                                         4   
                              CDLINKS-NoPOL                                           4   
                              CO_BAU                                                  4   
                              CO_CurPol                                               4   
                              CO_GPP                                                  4   
                              EN_NP_BL                                                4   
                              EN_NP_CurPol                                            4   
AIM/Hub-China 2.2             EN_NDCUnc2030_-100pc2050                                4   
                              EN_NDCUnc2030_-30pc2050                                 4   
                              EN_NDCUnc2030_-40pc2050                                 4   
                              EN_NDCUnc2030_-50pc2050                                 4   
                              EN_NDCUnc2030_-60pc2050                       

In [7]:
pd.options.display.max_rows = 1500

incompatibility_flagged = df[[
    "scenario_provider", "scenario", "scenario_type","sector", "technology",   "scenario_geography",
    "incompatible_var_cost","incompatible_fixed_cost"]].drop_duplicates()


incompatibility_flagged = incompatibility_flagged.groupby(["scenario_provider", "scenario"]).agg(
    n_techs = ("technology", "nunique"),
    n_regions = ("scenario_geography", "nunique"),
    incompatible_var_cost = ("incompatible_var_cost", "sum"),
    incompatible_fixed_cost = ("incompatible_fixed_cost", "sum"),
).assign(
    n_tech_regions = lambda x: x["n_techs"] * x["n_regions"],
    perc_incompatible_fixed_cost = lambda x: x["incompatible_fixed_cost"] / x["n_tech_regions"],
    perc_incompatible_var_cost = lambda x: x["incompatible_var_cost"] / x["n_tech_regions"]
)
print(incompatibility_flagged.shape)
# incompatibility_flagged.query("incompatible_fixed_cost < n_tech_regions/2")
# incompatibility_flagged.query("perc_incompatible_fixed_cost <= 0.25 & perc_incompatible_var_cost <= 0.25")
incompatibility_flagged.query("perc_incompatible_var_cost <= 0.25")

(1262, 7)


n_techs  \
scenario_provider             scenario                                                    
AIM/CGE 2.2                   EN_INDCi2030_1000f                                      4   
                              EN_INDCi2030_1200                                       4   
                              EN_INDCi2030_1200f                                      4   
                              EN_INDCi2030_1400                                       4   
                              EN_INDCi2030_1400f                                      4   
                              EN_INDCi2030_1600                                       4   
                              EN_INDCi2030_1600f                                      4   
                              EN_INDCi2030_1800                                       4   
                              EN_INDCi2030_1800f                                      4   
                              EN_INDCi2030_800f                                       4   
                              EN_INDCi2030_900f                                       4   
                              EN_INDCi2100                                            4   
                              EN_NPi2020_1000                                         4   
                              EN_NPi2020_1000f                                        4   
                              EN_NPi2020_1200                                         4   
                              EN_NPi2020_1200f                                        4   
                              EN_NPi2020_1400                                         4   
                              EN_NPi2020_1400f                                        4   
                              EN_NPi2020_1600                                         4   
                              EN_NPi2020_1600f                                        4   
                              EN_NPi2020_1800                                         4   
                              EN_NPi2020_1800f                                        4   
                              EN_NPi2020_300f                                         4   
                              EN_NPi2020_400f                                         4   
                              EN_NPi2020_500f                                         4   
                              EN_NPi2020_600                                          4   
                              EN_NPi2020_600f                                         4   
                              EN_NPi2020_700                                          4   
                              EN_NPi2020_700f                                         4   
                              EN_NPi2020_800                                          4   
                              EN_NPi2020_800f                                         4   
                              EN_NPi2020_900                                          4   
                              EN_NPi2020_900f                                         4   
                              EN_NPi2100                                              4   
AIM/CGE-Korea 2.1             CO_2Deg2020                                             4   
                              CO_2Deg2030                                             4   
                              CO_BAU                                                  4   
                              CO_Bridge                                               4   
                              CO_CurPol                                               4   
                              CO_GPP                                                  4   
                              CO_LowCarbon                                            4   
                              CO_NDCMCS                                               4   
AIM/Enduse-Japan 2.1          CDLINKS-NDC                                   